# Bersihkan Duplikat Dataset untuk Persiapan Training

Struktur folder yang diasumsikan: `root/<nama_kelas>/data/*.jpg`

Alur kerja:
1. Exact duplicate (hash SHA256 sama) dalam kelas yang sama -> hapus salah satu, sisakan satu file per grup.
2. Near duplicate (perceptual hash, jarak Hamming = 0) dalam kelas yang sama -> hapus salah satu, sisakan satu file per grup.
3. Duplikat (exact maupun near) yang lintas kelas -> hapus SEMUA file dalam grup, tidak ada yang disisakan, karena label yang benar tidak bisa ditentukan otomatis dari kemiripan visual.

EXECUTE = True secara default, artinya sel penghapusan langsung menghapus file sungguhan begitu dijalankan. Semua yang dihapus tetap tercatat di file log CSV untuk jejak audit.

In [2]:
import csv
import hashlib
import itertools
import os
from collections import defaultdict

from PIL import Image
import imagehash

EKSTENSI_VALID = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tiff"}

## 1. Atur parameter

In [3]:
ROOT = "archive"   # folder root dataset, isinya folder per kelas
EXECUTE = True      # True = benar benar menghapus file. Ganti ke False kalau mau dry run dulu.
LOG_CSV = "log_hapus_duplikat.csv"

## 2. Fungsi bantu

In [4]:
def daftar_gambar(root):
    hasil = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            if os.path.splitext(f)[1].lower() in EKSTENSI_VALID:
                hasil.append(os.path.join(dirpath, f))
    return hasil


def ambil_kelas(path, root):
    """Nama folder tingkat pertama di bawah root dianggap sebagai kelas."""
    rel = os.path.relpath(path, root)
    return rel.split(os.sep)[0]


def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def cek_exact_duplicate(paths):
    grup = defaultdict(list)
    for p in paths:
        try:
            grup[hash_file(p)].append(p)
        except OSError as e:
            print(f"Gagal baca {p}: {e}")
    return [v for v in grup.values() if len(v) > 1]


def hitung_phash(paths):
    hasil = {}
    for p in paths:
        try:
            with Image.open(p) as img:
                hasil[p] = imagehash.phash(img)
        except Exception as e:
            print(f"Gagal proses {p}: {e}")
    return hasil


def cek_near_duplicate_jarak0(paths):
    """Kembalikan grup near duplicate dengan jarak Hamming = 0,
    dikelompokkan lewat union find supaya A~B~C jadi satu grup,
    bukan sekadar pasangan lepas."""
    hash_map = hitung_phash(paths)
    items = list(hash_map.items())

    parent = {p: p for p, _ in items}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for (path_a, hash_a), (path_b, hash_b) in itertools.combinations(items, 2):
        if hash_a - hash_b == 0:
            union(path_a, path_b)

    grup = defaultdict(list)
    for p, _ in items:
        grup[find(p)].append(p)
    return [v for v in grup.values() if len(v) > 1]


def pisah_sama_dan_lintas_kelas(grup_list, root):
    sama_kelas = []
    lintas_kelas = []
    for grup in grup_list:
        kelas_set = set(ambil_kelas(p, root) for p in grup)
        if len(kelas_set) == 1:
            sama_kelas.append(grup)
        else:
            lintas_kelas.append(grup)
    return sama_kelas, lintas_kelas


def hapus_grup_sama_kelas(grup_list, execute, log_writer, tipe):
    """Sisakan satu file per grup (yang pertama secara alfabetis), hapus sisanya.
    File yang sudah lebih dulu terhapus oleh proses grup lain (exact duplicate
    dan near duplicate jarak 0 sering tumpang tindih) dilewati, bukan dianggap
    error, supaya tidak ada percobaan hapus dobel."""
    total_dihapus = 0
    for grup in grup_list:
        existing = [p for p in grup if (not execute) or os.path.exists(p)]
        if len(existing) < 2:
            continue
        grup_urut = sorted(existing)
        disimpan = grup_urut[0]
        dihapus = grup_urut[1:]
        for p in dihapus:
            log_writer.writerow([tipe, "sama_kelas", p, disimpan, "hapus" if execute else "dry_run"])
            if execute:
                try:
                    os.remove(p)
                except OSError as e:
                    print(f"Gagal hapus {p}: {e}")
            total_dihapus += 1
    return total_dihapus


def hapus_grup_lintas_kelas(grup_list, execute, log_writer, tipe):
    """Hapus SEMUA file dalam grup, tidak ada yang disisakan.
    File yang sudah lebih dulu terhapus oleh proses grup lain dilewati."""
    total_dihapus = 0
    for grup in grup_list:
        existing = [p for p in grup if (not execute) or os.path.exists(p)]
        for p in sorted(existing):
            log_writer.writerow([tipe, "lintas_kelas", p, "", "hapus" if execute else "dry_run"])
            if execute:
                try:
                    os.remove(p)
                except OSError as e:
                    print(f"Gagal hapus {p}: {e}")
            total_dihapus += 1
    return total_dihapus

## 3. Kumpulkan file dan deteksi duplikat

In [5]:
paths = daftar_gambar(ROOT)
print(f"Ditemukan {len(paths)} file gambar di {ROOT}")

grup_exact = cek_exact_duplicate(paths)
sama_exact, lintas_exact = pisah_sama_dan_lintas_kelas(grup_exact, ROOT)
print(f"\nExact duplicate: {len(grup_exact)} grup total")
print(f"  Sama kelas : {len(sama_exact)} grup (sisakan satu file per grup)")
print(f"  Lintas kelas: {len(lintas_exact)} grup (hapus semua file dalam grup)")

grup_near = cek_near_duplicate_jarak0(paths)
sama_near, lintas_near = pisah_sama_dan_lintas_kelas(grup_near, ROOT)
print(f"\nNear duplicate (jarak 0): {len(grup_near)} grup total")
print(f"  Sama kelas : {len(sama_near)} grup (sisakan satu file per grup)")
print(f"  Lintas kelas: {len(lintas_near)} grup (hapus semua file dalam grup)")

Ditemukan 14376 file gambar di archive

Exact duplicate: 0 grup total
  Sama kelas : 0 grup (sisakan satu file per grup)
  Lintas kelas: 0 grup (hapus semua file dalam grup)

Near duplicate (jarak 0): 0 grup total
  Sama kelas : 0 grup (sisakan satu file per grup)
  Lintas kelas: 0 grup (hapus semua file dalam grup)


## 4. Hapus duplikat

Sama kelas: sisakan satu file, hapus sisanya.
Lintas kelas: hapus seluruh file dalam grup, tidak ada yang disisakan.

EXECUTE dikontrol dari sel parameter di atas. Semua yang dihapus tercatat di LOG_CSV.

In [6]:
mode = "EXECUTE (file akan benar benar dihapus)" if EXECUTE else "DRY RUN (tidak ada file yang dihapus)"
print(f"Mode: {mode}\n")

with open(LOG_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["tipe_duplikat", "cakupan", "file_dihapus", "file_disimpan", "status"])

    total_exact_sama = hapus_grup_sama_kelas(sama_exact, EXECUTE, writer, "exact")
    total_near_sama = hapus_grup_sama_kelas(sama_near, EXECUTE, writer, "near_duplicate")
    total_exact_lintas = hapus_grup_lintas_kelas(lintas_exact, EXECUTE, writer, "exact")
    total_near_lintas = hapus_grup_lintas_kelas(lintas_near, EXECUTE, writer, "near_duplicate")

kata = "dihapus" if EXECUTE else "akan dihapus"
print(f"Exact duplicate sama kelas, file {kata}: {total_exact_sama}")
print(f"Near duplicate sama kelas, file {kata}: {total_near_sama}")
print(f"Exact duplicate lintas kelas, file {kata}: {total_exact_lintas}")
print(f"Near duplicate lintas kelas, file {kata}: {total_near_lintas}")
print(f"\nTotal file {kata}: {total_exact_sama + total_near_sama + total_exact_lintas + total_near_lintas}")
print(f"Detail tercatat di: {LOG_CSV}")

Mode: EXECUTE (file akan benar benar dihapus)

Exact duplicate sama kelas, file dihapus: 0
Near duplicate sama kelas, file dihapus: 0
Exact duplicate lintas kelas, file dihapus: 0
Near duplicate lintas kelas, file dihapus: 0

Total file dihapus: 0
Detail tercatat di: log_hapus_duplikat.csv


## 5. Verifikasi hasil

In [7]:
sisa = daftar_gambar(ROOT)
print(f"Jumlah file gambar tersisa di {ROOT}: {len(sisa)}")
print(f"Jumlah file sebelum dibersihkan: {len(paths)}")
print(f"Selisih: {len(paths) - len(sisa)}")

Jumlah file gambar tersisa di archive: 14376
Jumlah file sebelum dibersihkan: 14376
Selisih: 0
